<a href="https://colab.research.google.com/github/Yogskiy/big-data-kelompok-7-weather/blob/main/Gathering_Data_Cuaca_Jabar_Kelompok7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Wrangling — Gathering Data
## Analisis dan Prediksi Pola Cuaca Jawa Barat untuk Mitigasi Banjir dan Dukungan Sektor Pertanian

**Mata Kuliah:** Big Data
**Kelompok:** 7
**Anggota:**
1. Hilarius Yoga Baskoro — 101032300159
2. Wildan Janitra Ramadhan — 101032300140

**Tugas Minggu 3:** Fiksasi Dataset + Data Wrangling tahap *Gathering Data*

**Tanggal pengerjaan:** 30 September 2026

> Notebook ini menjalankan tahap **gathering** saja: mengambil data mentah dari sumber, menggabungkannya, memverifikasi strukturnya, lalu menyimpan dataset final. Pembersihan (*assessing* & *cleaning*) dikerjakan pada tugas minggu berikutnya sehingga dataset yang dihasilkan di sini sengaja masih apa adanya.

---
## 1. Deskripsi dan Fiksasi Dataset

### 1.1 Identitas dataset

| Aspek | Keterangan |
| --- | --- |
| Nama dataset | Hourly Weather Forecast Archive — 12 Kota/Kabupaten Jawa Barat |
| Penyedia | Open-Meteo (open-source weather API, non-komersial gratis tanpa API key) |
| Endpoint | `https://historical-forecast-api.open-meteo.com/v1/forecast` |
| Dokumentasi | https://open-meteo.com/en/docs/historical-forecast-api |
| Lisensi | CC BY 4.0 (atribusi wajib) |
| Periode | 1 Januari 2024 – 31 Desember 2025 |
| Resolusi | Per jam (hourly), zona waktu Asia/Jakarta (WIB) |
| Cakupan | 12 titik koordinat kota/kabupaten di Provinsi Jawa Barat |
| Estimasi baris | 12 × 17.544 jam = **210.528 baris** |
| Estimasi kolom | 19 kolom asal + 4 kolom identitas lokasi + 3 kolom metadata = **26 kolom** |
| Target/label | Tidak ada label bawaan. Untuk pemodelan nanti, `precipitation` (regresi curah hujan) atau `weather_code` (klasifikasi kondisi cuaca) diposisikan sebagai target. |

Perhitungan jumlah jam: 2024 adalah tahun kabisat (366 hari × 24 = 8.784 jam), 2025 memiliki 365 hari (× 24 = 8.760 jam). Totalnya 17.544 jam per kota.

### 1.2 Alasan pemilihan dataset

1. **Relevan dengan permasalahan.** Keputusan mitigasi banjir dan penjadwalan tanam bersifat operasional dan berjangka pendek, sehingga menuntut data pada resolusi per jam. Data iklim yang umum tersedia berhenti pada agregat bulanan yang terlalu kasar untuk menangkap hujan ekstrem yang berlangsung beberapa jam saja.
2. **Memenuhi kriteria Big Data secara wajar.** Volume 210 ribu baris diperoleh dari satu proses tarik, dan datanya bertambah 24 baris per kota per hari (velocity) karena endpoint yang sama terus diperbarui.
3. **Dapat diakses secara terprogram dan reproducible.** API gratis, tanpa kunci, dan parameternya eksplisit, sehingga proses gathering bisa diulang siapa pun dengan hasil identik — ini poin penting untuk pertanggungjawaban akademik.
4. **Wilayah Jawa Barat dipilih karena homogen secara administratif tetapi heterogen secara geografis** — dataset mencakup dataran tinggi (Bandung, Garut), pesisir utara (Cirebon, Bekasi), dan zona transisi (Sukabumi, Cianjur) — sehingga perbandingan antarkota tetap bermakna.

### 1.3 Catatan penting tentang pemilihan endpoint

Open-Meteo menyediakan beberapa arsip yang berbeda. Variabel yang kelompok kami butuhkan — `precipitation_probability`, `visibility`, `weather_code`, serta suhu tanah pada kedalaman 0 cm, 6 cm, dan 18 cm — **tidak tersedia** pada *Historical Weather API* (arsip reanalysis ERA5, host `archive-api.open-meteo.com`). Variabel-variabel itu hanya ada pada keluarga *Forecast API*.

Karena itu kami memakai **Historical Forecast API** (`historical-forecast-api.open-meteo.com`), yaitu arsip dari prakiraan yang pernah diterbitkan: potongan jam-jam awal setiap model run dirangkai menjadi satu deret waktu per jam yang berkesinambungan. Konsekuensi yang kami nyatakan terbuka di laporan: **data ini adalah keluaran model prakiraan, bukan hasil pengukuran stasiun BMKG**, dan versi model dapat berubah sepanjang arsip. Rentang 2024–2025 aman karena arsip ini tersedia sejak 2021/2022.

### 1.4 Daftar variabel yang difiksasi (19 kolom asal)

| No | Nama variabel API | Satuan | Keterangan singkat |
| --- | --- | --- | --- |
| 1 | `time` | ISO 8601 | Cap waktu per jam, waktu lokal WIB |
| 2 | `temperature_2m` | °C | Suhu udara pada ketinggian 2 m |
| 3 | `relative_humidity_2m` | % | Kelembapan relatif pada 2 m |
| 4 | `dew_point_2m` | °C | Titik embun pada 2 m |
| 5 | `precipitation` | mm | Total presipitasi (hujan + showers + salju) |
| 6 | `rain` | mm | Hujan dari sistem awan berskala luas |
| 7 | `showers` | mm | Hujan konvektif (lokal, intens, singkat) |
| 8 | `snowfall` | cm | Salju — diperkirakan bernilai 0 di Indonesia |
| 9 | `snow_depth` | m | Ketebalan salju — diperkirakan 0/kosong di Indonesia |
| 10 | `precipitation_probability` | % | Peluang terjadinya presipitasi |
| 11 | `surface_pressure` | hPa | Tekanan udara di permukaan |
| 12 | `cloud_cover` | % | Tutupan awan total |
| 13 | `visibility` | m | Jarak pandang |
| 14 | `vapour_pressure_deficit` | kPa | Defisit tekanan uap, indikator stres air tanaman |
| 15 | `wind_speed_10m` | km/jam | Kecepatan angin pada 10 m |
| 16 | `weather_code` | kode WMO | Kode kondisi cuaca (kategorik, bukan numerik) |
| 17 | `soil_temperature_0cm` | °C | Suhu permukaan tanah |
| 18 | `soil_temperature_6cm` | °C | Suhu tanah kedalaman 6 cm |
| 19 | `soil_temperature_18cm` | °C | Suhu tanah kedalaman 18 cm |

Dua variabel salju (`snowfall`, `snow_depth`) sengaja tetap ditarik agar struktur dataset konsisten dengan spesifikasi yang difiksasi. Nilainya yang konstan nol akan didokumentasikan sebagai temuan pada tahap *assessing*, bukan dihapus diam-diam di tahap gathering.

---
## 2. Import Library

Empat kelompok pustaka yang dipakai:

- `requests` beserta `HTTPAdapter` dan `Retry` — melakukan permintaan HTTP ke API sekaligus mengulang otomatis bila server membalas error sementara atau membatasi laju permintaan.
- `pandas` — struktur DataFrame, penggabungan, dan penyimpanan hasil.
- `pathlib`, `time`, `json` — pengelolaan berkas, jeda antar permintaan, dan inspeksi respons mentah.
- `google.colab.drive` — menyimpan hasil unduhan ke Google Drive agar tidak hilang ketika runtime Colab terputus.

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("pandas :", pd.__version__)
print("requests:", requests.__version__)

pandas : 2.2.3
requests: 2.32.4


---
## 3. Konfigurasi Parameter Gathering

Seluruh parameter proses dikumpulkan dalam satu sel agar percobaan ulang cukup mengubah satu tempat dan tidak ada nilai yang tersebar di dalam kode.

**Koordinat.** Setiap kota diwakili satu titik koordinat pusat kota. Ini adalah penyederhanaan yang perlu disebut di laporan: API mengembalikan nilai dari sel grid model terdekat, sehingga angkanya mewakili kondisi di sekitar titik tersebut, bukan rata-rata seluruh wilayah administratif.

**Zona waktu.** Parameter `timezone=Asia/Jakarta` membuat API langsung mengembalikan cap waktu dalam WIB. Tanpa parameter ini data akan berbasis UTC dan seluruh analisis pola harian (misalnya jam puncak hujan) akan bergeser tujuh jam.

In [ ]:
# --- Sumber data -----------------------------------------------------------
ENDPOINT = "https://historical-forecast-api.open-meteo.com/v1/forecast"

# --- Rentang waktu ---------------------------------------------------------
START_DATE = "2024-01-01"
END_DATE   = "2025-12-31"
TIMEZONE   = "Asia/Jakarta"

# --- 12 titik lokasi di Provinsi Jawa Barat --------------------------------
# Format: "Nama": (latitude, longitude)
KOTA = {
    "Bandung":     (-6.9175, 107.6191),
    "Bogor":       (-6.5950, 106.8166),
    "Depok":       (-6.4025, 106.7942),
    "Bekasi":      (-6.2383, 106.9756),
    "Cimahi":      (-6.8722, 107.5425),
    "Cirebon":     (-6.7320, 108.5523),
    "Sukabumi":    (-6.9277, 106.9300),
    "Tasikmalaya": (-7.3274, 108.2207),
    "Banjar":      (-7.3700, 108.5342),
    "Garut":       (-7.2145, 107.9087),
    "Cianjur":     (-6.8201, 107.1423),
    "Purwakarta":  (-6.5569, 107.4433),
}

# --- 18 variabel cuaca per jam (kolom `time` otomatis disertakan API) ------
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "showers",
    "snowfall",
    "snow_depth",
    "precipitation_probability",
    "surface_pressure",
    "cloud_cover",
    "visibility",
    "vapour_pressure_deficit",
    "wind_speed_10m",
    "weather_code",
    "soil_temperature_18cm",
    "soil_temperature_6cm",
    "soil_temperature_0cm",
]

# --- Ekspektasi jumlah baris untuk verifikasi nanti ------------------------
JAM_PER_KOTA   = 366 * 24 + 365 * 24          # 2024 kabisat + 2025 = 17.544
BARIS_HARAPAN  = JAM_PER_KOTA * len(KOTA)     # 210.528

print(f"Jumlah lokasi        : {len(KOTA)}")
print(f"Jumlah variabel cuaca: {len(HOURLY_VARS)}")
print(f"Jam per kota         : {JAM_PER_KOTA:,}")
print(f"Total baris harapan  : {BARIS_HARAPAN:,}")

Jumlah lokasi        : 12
Jumlah variabel cuaca: 18
Jam per kota         : 17,544
Total baris harapan  : 210,528


---
## 4. Menyiapkan Penyimpanan (Google Drive)

Runtime Colab bersifat sementara — seluruh isi `/content` hilang saat sesi berakhir. Karena satu proses tarik penuh memakan waktu beberapa menit dan membebani API, hasilnya disimpan ke Google Drive supaya tidak perlu diunduh ulang setiap kali notebook dijalankan.

Sel di bawah dibuat toleran: jika notebook dijalankan di luar Colab (misalnya Jupyter lokal), proses tetap berlanjut dengan folder lokal.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path("/content/drive/MyDrive/cuaca-jabar")
except Exception as e:
    print(f"Google Drive tidak tersedia ({type(e).__name__}), memakai penyimpanan lokal.")
    BASE_DIR = Path("./cuaca-jabar")

RAW_DIR   = BASE_DIR / "01_data_mentah"      # satu CSV per kota (checkpoint)
FINAL_DIR = BASE_DIR / "02_dataset_final"    # dataset gabungan

for d in (RAW_DIR, FINAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Folder data mentah  :", RAW_DIR)
print("Folder dataset final:", FINAL_DIR)

Mounted at /content/drive
Folder data mentah  : /content/drive/MyDrive/cuaca-jabar/01_data_mentah
Folder dataset final: /content/drive/MyDrive/cuaca-jabar/02_dataset_final


---
## 5. Fungsi Pengambilan Data

Tiga hal yang ditangani fungsi ini, masing-masing menjawab satu risiko nyata:

1. **Retry otomatis.** API gratis Open-Meteo membatasi laju permintaan dan membalas kode `429` bila kuota per menit terlampaui. `Retry` dengan `backoff_factor=2` akan menunggu 2, 4, 8, 16 detik sebelum mencoba lagi, sehingga permintaan yang ditolak sementara tidak membuat seluruh proses gagal.
2. **Penanganan error aplikasi.** Bila parameter salah, Open-Meteo membalas HTTP 400 dengan badan JSON `{"error": true, "reason": "..."}`. Pesan itu dibaca dan dilemparkan sebagai exception agar penyebabnya terlihat jelas, bukan sekadar "400 Bad Request".
3. **Pengayaan identitas.** Respons API hanya berisi deret waktu tanpa nama kota. Kolom `city`, `latitude`, `longitude`, dan `elevation_m` ditambahkan di sini supaya setiap baris tetap bisa ditelusuri asalnya setelah 12 tabel digabungkan.

In [ ]:
def buat_session(total_retry: int = 5) -> requests.Session:
    '''Session HTTP dengan retry bertahap untuk error sementara dan rate limit.'''
    retry = Retry(
        total=total_retry,
        backoff_factor=2,                                  # jeda 2, 4, 8, 16 detik
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


SESSION = buat_session()


def ambil_data_kota(nama: str, lat: float, lon: float, timeout: int = 120) -> pd.DataFrame:
    '''Tarik deret waktu per jam satu kota dan kembalikan sebagai DataFrame.'''
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(HOURLY_VARS),
        "timezone": TIMEZONE,
    }

    resp = SESSION.get(ENDPOINT, params=params, timeout=timeout)

    # Open-Meteo menaruh alasan kegagalan di dalam body JSON, bukan di status text.
    if resp.status_code != 200:
        try:
            alasan = resp.json().get("reason", resp.text[:300])
        except ValueError:
            alasan = resp.text[:300]
        raise RuntimeError(f"[{nama}] HTTP {resp.status_code} — {alasan}")

    payload = resp.json()

    df = pd.DataFrame(payload["hourly"])
    df.insert(0, "city", nama)
    df["latitude"]    = payload["latitude"]     # koordinat grid yang benar-benar dipakai model
    df["longitude"]   = payload["longitude"]
    df["elevation_m"] = payload["elevation"]

    return df, payload

---
## 6. Uji Coba Satu Kota

Sebelum menjalankan perulangan 12 kota, satu kota diuji lebih dulu. Tujuannya memastikan nama variabel, rentang tanggal, dan zona waktu sudah benar — jauh lebih murah daripada menemukan kesalahan setelah 12 permintaan besar terkirim.

Yang diperiksa pada langkah ini:
- jumlah baris harus tepat 17.544;
- cap waktu pertama `2024-01-01T00:00` dan terakhir `2025-12-31T23:00`;
- satuan setiap variabel sesuai spesifikasi yang difiksasi.

In [ ]:
df_uji, payload_uji = ambil_data_kota("Bandung", *KOTA["Bandung"])

print("Dimensi (baris, kolom):", df_uji.shape)
print("Cap waktu awal        :", df_uji['time'].iloc[0])
print("Cap waktu akhir       :", df_uji['time'].iloc[-1])
print("Zona waktu respons    :", payload_uji['timezone_abbreviation'])
print("Elevasi titik grid    :", payload_uji['elevation'], "m\n")

print("Satuan tiap variabel menurut API:")
print(json.dumps(payload_uji["hourly_units"], indent=2))

Dimensi (baris, kolom): (17544, 23)
Cap waktu awal        : 2024-01-01T00:00
Cap waktu akhir       : 2025-12-31T23:00
Zona waktu respons    : GMT+7
Elevasi titik grid    : 703.0 m

Satuan tiap variabel menurut API:
{
  "time": "iso8601",
  "temperature_2m": "\u00b0C",
  "relative_humidity_2m": "%",
  "dew_point_2m": "\u00b0C",
  "precipitation": "mm",
  "rain": "mm",
  "showers": "mm",
  "snowfall": "cm",
  "snow_depth": "m",
  "precipitation_probability": "%",
  "surface_pressure": "hPa",
  "cloud_cover": "%",
  "visibility": "m",
  "vapour_pressure_deficit": "kPa",
  "wind_speed_10m": "km/h",
  "weather_code": "wmo code",
  "soil_temperature_18cm": "\u00b0C",
  "soil_temperature_6cm": "\u00b0C",
  "soil_temperature_0cm": "\u00b0C"
}


In [ ]:
df_uji.head()

,city,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,rain,showers,snowfall,snow_depth,precipitation_probability,surface_pressure,cloud_cover,visibility,vapour_pressure_deficit,wind_speed_10m,weather_code,soil_temperature_18cm,soil_temperature_6cm,soil_temperature_0cm,latitude,longitude,elevation_m
0,Bandung,2024-01-01T00:00,20.2,93,18.9,0.0,0.0,0.0,0.0,0.0,NaN,933.5,100,24140.0,0.18,6.7,3,23.6,21.5,23.8,-6.924429,107.60539,703.0
1,Bandung,2024-01-01T01:00,19.8,93,18.7,0.0,0.0,0.0,0.0,0.0,NaN,932.8,100,24140.0,0.15,7.0,3,23.5,21.3,23.5,-6.924429,107.60539,703.0
2,Bandung,2024-01-01T02:00,19.8,92,18.5,0.0,0.0,0.0,0.0,0.0,NaN,932.4,100,24140.0,0.18,6.4,3,23.3,21.0,22.9,-6.924429,107.60539,703.0
3,Bandung,2024-01-01T03:00,20.0,90,18.3,0.0,0.0,0.0,0.0,0.0,NaN,932.3,99,24140.0,0.24,4.6,3,23.2,20.8,22.3,-6.924429,107.60539,703.0
4,Bandung,2024-01-01T04:00,20.8,86,18.4,0.0,0.0,0.0,0.0,0.0,NaN,932.5,100,24140.0,0.34,3.1,3,23.0,20.5,22.3,-6.924429,107.60539,703.0


---
## 7. Gathering 12 Kota

Perulangan dijalankan dengan tiga pengaman:

- **Checkpoint per kota.** Setiap kota langsung disimpan sebagai CSV tersendiri. Bila sel terputus di tengah jalan, kota yang sudah berhasil tidak perlu ditarik ulang — sel ini melewatinya secara otomatis.
- **Jeda antar permintaan.** `time.sleep(2)` menjaga jarak antar panggilan agar tidak memicu pembatasan laju. Open-Meteo menghitung bobot permintaan dari jumlah variabel dikali rentang hari, sehingga 18 variabel selama 730 hari tergolong permintaan berat meski hanya satu panggilan.
- **Pencatatan kegagalan.** Kota yang gagal dikumpulkan dalam daftar `gagal` dan dilaporkan di akhir, sehingga proses tidak berhenti total hanya karena satu titik bermasalah.

In [ ]:
gagal = []
mulai = time.time()

for i, (nama, (lat, lon)) in enumerate(KOTA.items(), start=1):
    berkas = RAW_DIR / f"{nama.lower()}_2024_2025.csv"

    if berkas.exists():
        print(f"[{i:2}/{len(KOTA)}] {nama:<12} dilewati — berkas checkpoint sudah ada")
        continue

    try:
        df_kota, _ = ambil_data_kota(nama, lat, lon)
        df_kota.to_csv(berkas, index=False)
        print(f"[{i:2}/{len(KOTA)}] {nama:<12} OK  {df_kota.shape[0]:,} baris  ->  {berkas.name}")
    except Exception as e:
        gagal.append((nama, str(e)))
        print(f"[{i:2}/{len(KOTA)}] {nama:<12} GAGAL — {e}")

    time.sleep(2)

print(f"\nDurasi total: {time.time() - mulai:.1f} detik")
print(f"Berhasil: {len(KOTA) - len(gagal)} kota | Gagal: {len(gagal)} kota")
if gagal:
    print("Daftar kota yang perlu ditarik ulang:", [g[0] for g in gagal])

[ 1/12] Bandung      OK  17,544 baris  ->  bandung_2024_2025.csv
[ 2/12] Bogor        OK  17,544 baris  ->  bogor_2024_2025.csv
[ 3/12] Depok        OK  17,544 baris  ->  depok_2024_2025.csv
[ 4/12] Bekasi       OK  17,544 baris  ->  bekasi_2024_2025.csv
[ 5/12] Cimahi       OK  17,544 baris  ->  cimahi_2024_2025.csv
[ 6/12] Cirebon      OK  17,544 baris  ->  cirebon_2024_2025.csv
[ 7/12] Sukabumi     OK  17,544 baris  ->  sukabumi_2024_2025.csv
[ 8/12] Tasikmalaya  OK  17,544 baris  ->  tasikmalaya_2024_2025.csv
[ 9/12] Banjar       OK  17,544 baris  ->  banjar_2024_2025.csv
[10/12] Garut        OK  17,544 baris  ->  garut_2024_2025.csv
[11/12] Cianjur      OK  17,544 baris  ->  cianjur_2024_2025.csv
[12/12] Purwakarta   OK  17,544 baris  ->  purwakarta_2024_2025.csv

Durasi total: 64.9 detik
Berhasil: 12 kota | Gagal: 0 kota


---
## 8. Penggabungan Data (Concat)

Dua belas berkas CSV memiliki skema yang identik — kolom, urutan, dan tipe datanya sama — karena semuanya berasal dari satu endpoint dengan parameter yang sama. Penggabungan yang tepat untuk kasus ini adalah **penumpukan vertikal (`pd.concat`)**, bukan `merge` atau `join`, sebab yang bertambah adalah jumlah observasi, bukan jumlah atribut.

`ignore_index=True` dipakai agar indeks baris disusun ulang dari 0 sampai 210.527; tanpa itu indeks 0–17.543 akan berulang dua belas kali dan menyulitkan penelusuran baris.

Pembacaan dilakukan ulang dari berkas checkpoint (bukan dari variabel di memori) supaya sel ini tetap bisa dijalankan sendiri setelah runtime di-restart.

In [ ]:
berkas_csv = sorted(RAW_DIR.glob("*_2024_2025.csv"))
print(f"Berkas terdeteksi: {len(berkas_csv)}")

potongan = []
for b in berkas_csv:
    d = pd.read_csv(b)
    potongan.append(d)
    print(f"  {b.name:<30} {d.shape[0]:>7,} baris x {d.shape[1]:>2} kolom")

df_all = pd.concat(potongan, ignore_index=True)

print(f"\nHasil concat: {df_all.shape[0]:,} baris x {df_all.shape[1]} kolom")
print(f"Ekspektasi  : {BARIS_HARAPAN:,} baris")
print("Status      :", "SESUAI" if len(df_all) == BARIS_HARAPAN else "PERLU DIPERIKSA")

Berkas terdeteksi: 12
  bandung_2024_2025.csv           17,544 baris x 23 kolom
  banjar_2024_2025.csv            17,544 baris x 23 kolom
  bekasi_2024_2025.csv            17,544 baris x 23 kolom
  bogor_2024_2025.csv             17,544 baris x 23 kolom
  cianjur_2024_2025.csv           17,544 baris x 23 kolom
  cimahi_2024_2025.csv            17,544 baris x 23 kolom
  cirebon_2024_2025.csv           17,544 baris x 23 kolom
  depok_2024_2025.csv             17,544 baris x 23 kolom
  garut_2024_2025.csv             17,544 baris x 23 kolom
  purwakarta_2024_2025.csv        17,544 baris x 23 kolom
  sukabumi_2024_2025.csv          17,544 baris x 23 kolom
  tasikmalaya_2024_2025.csv       17,544 baris x 23 kolom

Hasil concat: 210,528 baris x 23 kolom
Ekspektasi  : 210,528 baris
Status      : SESUAI


---
## 9. Penggabungan Atribut Lokasi (Merge / Join)

Tahap sebelumnya menyatukan data yang sejenis. Di sini dilakukan penggabungan jenis kedua: **menambahkan atribut** dari tabel referensi ke tabel utama menggunakan `pd.merge` dengan kunci `city`.

Tabel referensi berisi status administratif dan pengelompokan wilayah. Gunanya konkret untuk analisis lanjutan — perbandingan pola hujan antara zona pesisir utara dan dataran tinggi Priangan jauh lebih mudah dilakukan lewat satu kolom `region` daripada menuliskan daftar kota berulang kali di setiap query.

Dipakai `how="left"` agar tidak ada satu pun baris cuaca yang hilang seandainya tabel referensi kurang lengkap. Hasil merge diverifikasi dengan memeriksa nilai kosong pada kolom hasil gabungan — cara paling cepat mendeteksi ketidakcocokan penulisan nama kota.

In [ ]:
metadata_kota = pd.DataFrame([
    # city,          status administratif, wilayah pengelompokan
    ("Bandung",      "Kota",      "Bandung Raya"),
    ("Cimahi",       "Kota",      "Bandung Raya"),
    ("Bogor",        "Kota",      "Bodebek"),
    ("Depok",        "Kota",      "Bodebek"),
    ("Bekasi",       "Kota",      "Bodebek"),
    ("Cirebon",      "Kota",      "Pantura"),
    ("Sukabumi",     "Kota",      "Jawa Barat Selatan"),
    ("Cianjur",      "Kabupaten", "Jawa Barat Tengah"),
    ("Tasikmalaya",  "Kota",      "Priangan Timur"),
    ("Banjar",       "Kota",      "Priangan Timur"),
    ("Garut",        "Kabupaten", "Priangan Timur"),
    ("Purwakarta",   "Kabupaten", "Purwasuka"),
], columns=["city", "status_administratif", "region"])

metadata_kota["province"] = "Jawa Barat"

df_all = df_all.merge(metadata_kota, on="city", how="left")

# Verifikasi merge: tidak boleh ada baris yang gagal mendapat pasangan.
tidak_cocok = df_all["region"].isna().sum()
print(f"Baris tanpa pasangan metadata: {tidak_cocok}")
print(f"Dimensi setelah merge        : {df_all.shape[0]:,} baris x {df_all.shape[1]} kolom\n")

print(df_all.groupby(["region", "city"]).size().rename("jumlah_baris").to_frame())

Baris tanpa pasangan metadata: 0
Dimensi setelah merge        : 210,528 baris x 26 kolom

                                jumlah_baris
region             city                     
Bandung Raya       Bandung             17544
                   Cimahi              17544
Bodebek            Bekasi              17544
                   Bogor               17544
                   Depok               17544
Jawa Barat Selatan Sukabumi            17544
Jawa Barat Tengah  Cianjur             17544
Pantura            Cirebon             17544
Priangan Timur     Banjar              17544
                   Garut               17544
                   Tasikmalaya         17544
Purwasuka          Purwakarta          17544


---
## 10. Pemeriksaan Struktur Awal Dataset

Tahap gathering belum boleh dinyatakan selesai sebelum hasilnya diperiksa. Lima pemeriksaan berikut adalah pemeriksaan **integritas pengumpulan**, bukan pembersihan data — yang dicari adalah bukti bahwa data terkumpul lengkap dan utuh, bukan memperbaiki nilainya.

1. **Struktur dan tipe data** — memastikan seluruh kolom terbaca dan tipe datanya masuk akal.
2. **Kelengkapan per kota** — setiap kota harus memiliki tepat 17.544 baris. Kekurangan menandakan ada jam yang hilang dari respons API.
3. **Duplikasi kunci** — kombinasi `city` + `time` harus unik. Duplikat menandakan satu kota terunduh atau tergabung dua kali.
4. **Kontinuitas waktu** — selisih antar cap waktu berturut-turut harus konsisten satu jam, tanpa lompatan.
5. **Profil nilai kosong dan kolom konstan** — bukan untuk dibersihkan sekarang, melainkan untuk dicatat sebagai temuan awal di laporan.

In [ ]:
print("=== 10.1 Struktur dan tipe data ===")
df_all.info(show_counts=True)

=== 10.1 Struktur dan tipe data ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210528 entries, 0 to 210527
Data columns (total 26 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   city                       210528 non-null  object 
 1   time                       210528 non-null  object 
 2   temperature_2m             210528 non-null  float64
 3   relative_humidity_2m       210528 non-null  int64  
 4   dew_point_2m               210528 non-null  float64
 5   precipitation              210528 non-null  float64
 6   rain                       210528 non-null  float64
 7   showers                    210528 non-null  float64
 8   snowfall                   210528 non-null  float64
 9   snow_depth                 210528 non-null  float64
 10  precipitation_probability  176400 non-null  float64
 11  surface_pressure           210528 non-null  float64
 12  cloud_cover                210528 non-null  int64 

In [ ]:
print("=== 10.2 Kelengkapan baris per kota ===")
cek = df_all.groupby("city").size().rename("jumlah_baris").to_frame()
cek["harapan"] = JAM_PER_KOTA
cek["selisih"] = cek["jumlah_baris"] - cek["harapan"]
cek["status"]  = cek["selisih"].apply(lambda x: "LENGKAP" if x == 0 else "PERIKSA")
print(cek)

=== 10.2 Kelengkapan baris per kota ===
             jumlah_baris  harapan  selisih   status
city                                                
Bandung             17544    17544        0  LENGKAP
Banjar              17544    17544        0  LENGKAP
Bekasi              17544    17544        0  LENGKAP
Bogor               17544    17544        0  LENGKAP
Cianjur             17544    17544        0  LENGKAP
Cimahi              17544    17544        0  LENGKAP
Cirebon             17544    17544        0  LENGKAP
Depok               17544    17544        0  LENGKAP
Garut               17544    17544        0  LENGKAP
Purwakarta          17544    17544        0  LENGKAP
Sukabumi            17544    17544        0  LENGKAP
Tasikmalaya         17544    17544        0  LENGKAP


In [ ]:
print("=== 10.3 Duplikasi pasangan city + time ===")
jumlah_duplikat = df_all.duplicated(subset=["city", "time"]).sum()
print(f"Baris duplikat: {jumlah_duplikat}")

print("\n=== 10.4 Kontinuitas deret waktu ===")
df_all["time"] = pd.to_datetime(df_all["time"])
selisih = (
    df_all.sort_values(["city", "time"])
          .groupby("city")["time"]
          .diff()
          .dropna()
          .value_counts()
)
print(selisih)
print("\nInterpretasi: seluruh selisih bernilai 0 days 01:00:00 berarti tidak ada jam yang bolong.")
print(f"Rentang waktu: {df_all['time'].min()}  s.d.  {df_all['time'].max()}")

=== 10.3 Duplikasi pasangan city + time ===
Baris duplikat: 0

=== 10.4 Kontinuitas deret waktu ===
time
0 days 01:00:00    210516
Name: count, dtype: int64

Interpretasi: seluruh selisih bernilai 0 days 01:00:00 berarti tidak ada jam yang bolong.
Rentang waktu: 2024-01-01 00:00:00  s.d.  2025-12-31 23:00:00


In [ ]:
print("=== 10.5 Profil nilai kosong dan kolom konstan ===")
profil = pd.DataFrame({
    "tipe":          df_all.dtypes.astype(str),
    "nilai_kosong":  df_all.isna().sum(),
    "persen_kosong": (df_all.isna().mean() * 100).round(2),
    "nilai_unik":    df_all.nunique(dropna=True),
})
profil["catatan"] = profil.apply(
    lambda r: "SELURUHNYA KOSONG" if r["persen_kosong"] == 100
    else ("KONSTAN" if r["nilai_unik"] <= 1 else ""),
    axis=1,
)
print(profil)

print("\nKolom yang perlu dibahas di laporan sebagai temuan awal:")
print(list(profil[profil["catatan"] != ""].index) or "tidak ada")

=== 10.5 Profil nilai kosong dan kolom konstan ===
                                     tipe  nilai_kosong  persen_kosong  nilai_unik  catatan
city                               object             0           0.00          12         
time                       datetime64[ns]             0           0.00       17544         
temperature_2m                    float64             0           0.00         243         
relative_humidity_2m                int64             0           0.00          81         
dew_point_2m                      float64             0           0.00         220         
precipitation                     float64             0           0.00         213         
rain                              float64             0           0.00         213         
showers                           float64             0           0.00         156         
snowfall                          float64             0           0.00           1  KONSTAN
snow_depth                   

In [ ]:
print("=== 10.6 Ringkasan statistik variabel numerik utama ===")
kolom_inti = ["temperature_2m", "relative_humidity_2m", "precipitation",
              "surface_pressure", "cloud_cover", "wind_speed_10m",
              "soil_temperature_0cm"]
print(df_all[kolom_inti].describe().T.round(2))

=== 10.6 Ringkasan statistik variabel numerik utama ===
                         count    mean    std    min     25%     50%     75%     max
temperature_2m        210528.0   25.01   3.39   13.6   22.50   24.70    27.2    38.6
relative_humidity_2m  210528.0   82.13  14.21   19.0   73.00   86.00    94.0   100.0
precipitation         210528.0    0.33   1.23    0.0    0.00    0.00     0.1    29.0
surface_pressure      210528.0  973.11  30.68  922.1  940.97  977.05  1003.0  1014.7
cloud_cover           210528.0   82.35  29.54    0.0   77.00  100.00   100.0   100.0
wind_speed_10m        210528.0    5.49   3.70    0.0    2.90    4.60     7.2    30.8
soil_temperature_0cm  210528.0   26.39   4.73    9.4   23.00   25.50    29.4    48.7


---
## 11. Dataset Akhir Hasil Gathering

Dataset disimpan dalam dua format karena keduanya melayani kebutuhan berbeda:

- **CSV** — format penyerahan tugas, bisa dibuka siapa pun tanpa pustaka khusus, tetapi berukuran besar dan kehilangan informasi tipe data.
- **Parquet** — format kolomnar terkompresi. Ukurannya jauh lebih kecil, tipe data tersimpan utuh, dan pembacaannya cepat. Format inilah yang akan dipakai pada tahap pengolahan berikutnya.

Kolom diurutkan ulang lebih dahulu agar identitas lokasi dan waktu berada di depan, diikuti variabel cuaca. Urutan ini tidak mengubah isi data, hanya membuat dataset lebih mudah dibaca manusia.

In [ ]:
urutan_kolom = (
    ["city", "status_administratif", "region", "province",
     "latitude", "longitude", "elevation_m", "time"]
    + HOURLY_VARS
)
df_final = df_all[urutan_kolom].sort_values(["city", "time"]).reset_index(drop=True)

path_csv     = FINAL_DIR / "cuaca_jabar_12kota_2024_2025.csv"
path_parquet = FINAL_DIR / "cuaca_jabar_12kota_2024_2025.parquet"

df_final.to_csv(path_csv, index=False)
df_final.to_parquet(path_parquet, index=False)

print("DATASET FINAL HASIL GATHERING")
print("-" * 60)
print(f"Jumlah baris  : {df_final.shape[0]:,}")
print(f"Jumlah kolom  : {df_final.shape[1]}")
print(f"Jumlah lokasi : {df_final['city'].nunique()}")
print(f"Periode       : {df_final['time'].min()}  s.d.  {df_final['time'].max()}")
print(f"CSV           : {path_csv}  ({path_csv.stat().st_size / 1_048_576:.1f} MB)")
print(f"Parquet       : {path_parquet}  ({path_parquet.stat().st_size / 1_048_576:.1f} MB)")
print("-" * 60)

df_final.head(10)

DATASET FINAL HASIL GATHERING
------------------------------------------------------------
Jumlah baris  : 210,528
Jumlah kolom  : 26
Jumlah lokasi : 12
Periode       : 2024-01-01 00:00:00  s.d.  2025-12-31 23:00:00
CSV           : /content/drive/MyDrive/cuaca-jabar/02_dataset_final/cuaca_jabar_12kota_2024_2025.csv  (32.6 MB)
Parquet       : /content/drive/MyDrive/cuaca-jabar/02_dataset_final/cuaca_jabar_12kota_2024_2025.parquet  (3.1 MB)
------------------------------------------------------------


,city,status_administratif,region,province,latitude,longitude,elevation_m,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,rain,showers,snowfall,snow_depth,precipitation_probability,surface_pressure,cloud_cover,visibility,vapour_pressure_deficit,wind_speed_10m,weather_code,soil_temperature_18cm,soil_temperature_6cm,soil_temperature_0cm
0,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 00:00:00,20.2,93,18.9,0.0,0.0,0.0,0.0,0.0,NaN,933.5,100,24140.0,0.18,6.7,3,23.6,21.5,23.8
1,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 01:00:00,19.8,93,18.7,0.0,0.0,0.0,0.0,0.0,NaN,932.8,100,24140.0,0.15,7.0,3,23.5,21.3,23.5
2,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 02:00:00,19.8,92,18.5,0.0,0.0,0.0,0.0,0.0,NaN,932.4,100,24140.0,0.18,6.4,3,23.3,21.0,22.9
3,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 03:00:00,20.0,90,18.3,0.0,0.0,0.0,0.0,0.0,NaN,932.3,99,24140.0,0.24,4.6,3,23.2,20.8,22.3
4,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 04:00:00,20.8,86,18.4,0.0,0.0,0.0,0.0,0.0,NaN,932.5,100,24140.0,0.34,3.1,3,23.0,20.5,22.3
5,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 05:00:00,19.4,91,17.9,0.0,0.0,0.0,0.0,0.0,NaN,932.3,100,24140.0,0.21,4.8,3,22.8,20.3,22.9
6,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 06:00:00,19.7,91,18.2,0.0,0.0,0.0,0.0,0.0,NaN,933.0,100,24140.0,0.21,3.1,3,22.7,20.2,24.1
7,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 07:00:00,22.3,84,19.5,0.0,0.0,0.0,0.0,0.0,NaN,934.7,100,24140.0,0.42,2.5,3,22.6,21.0,26.2
8,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 08:00:00,23.9,81,20.4,0.0,0.0,0.0,0.0,0.0,NaN,935.2,100,24140.0,0.56,3.1,3,22.5,22.1,30.2
9,Bandung,Kota,Bandung Raya,Jawa Barat,-6.924429,107.60539,703.0,2024-01-01 09:00:00,25.0,79,21.0,0.0,0.0,0.0,0.0,0.0,NaN,935.4,100,24140.0,0.67,6.1,3,22.6,23.4,35.2


---
## 12. Ringkasan Proses Gathering

| Tahap | Tindakan | Keluaran |
| --- | --- | --- |
| 1 | Fiksasi sumber, periode, lokasi, dan 19 variabel | Spesifikasi dataset |
| 2 | Uji coba satu kota untuk validasi parameter | 17.544 baris terverifikasi |
| 3 | Penarikan 12 kota dengan retry dan checkpoint | 12 berkas CSV mentah |
| 4 | Penumpukan vertikal dengan `pd.concat` | 210.528 baris |
| 5 | Penggabungan atribut lokasi dengan `pd.merge` | 26 kolom |
| 6 | Verifikasi kelengkapan, duplikasi, dan kontinuitas waktu | Bukti integritas data |
| 7 | Penyimpanan format CSV dan Parquet | Dataset final |

### Kendala dan penanganannya

| Kendala | Dampak | Penanganan |
| --- | --- | --- |
| Variabel `precipitation_probability`, `visibility`, `weather_code`, dan suhu tanah berlapis (`0cm`, `6cm`, `18cm`) tidak tersedia pada endpoint *Historical Weather API* (ERA5 `archive-api.open-meteo.com`). | Permintaan API gagal atau kolom bernilai kosong seluruhnya jika tetap menggunakan endpoint arsip ERA5. | Beralih ke endpoint **Historical Forecast API** (`historical-forecast-api.open-meteo.com/v1/forecast`) yang menyimpan rangkaian arsip model prakiraan per jam secara lengkap untuk periode 2024–2025. |
| Pembatasan laju permintaan (*rate limiting* HTTP `429 Too Many Requests`) karena bobot penarikan 18 variabel × 731 hari per kota cukup besar. | Perulangan 12 kota berisiko terputus di tengah jalan dan kehilangan progres kota yang sudah berhasil diunduh. | Memasang `HTTPAdapter` dengan `Retry(total=5, backoff_factor=2)` untuk *exponential backoff*, memberi jeda `time.sleep(2)` antar-kota, serta menyimpan *atomic checkpoint* CSV per kota di `01_data_mentah/`. |
| Kolom `precipitation_probability` memiliki 34.128 nilai kosong (16,21%) pada sebagian periode tahun 2024, serta kolom `snowfall` dan `snow_depth` bernilai `0.0` konstan. | Dapat memicu bias atau *error* pada algoritma pemodelan jika langsung dimasukkan tanpa perlakuan khusus. | Dicatat sebagai temuan pemeriksaan awal (*initial data profiling*) tanpa dimanipulasi pada tahap *gathering*, untuk kemudian ditangani secara sistematis pada tahap *Assessing & Cleaning*. |

### Langkah berikutnya

Dataset ini masuk ke tahap *Assessing Data* pada minggu berikutnya: pemeriksaan kolom bernilai konstan, penetapan tipe data yang benar untuk `weather_code` sebagai variabel kategorik, serta penanganan nilai kosong bila ada.

---

**Atribusi sumber data:** Data cuaca bersumber dari Open-Meteo (https://open-meteo.com) dan digunakan di bawah lisensi CC BY 4.0.

---
## Lampiran A — Jalur Alternatif: Membaca Berkas CSV Hasil Unduhan Manual

Jika data diunduh lewat tombol *Download CSV* di situs Open-Meteo, berkasnya **tidak bisa langsung dibaca** `pd.read_csv` tanpa penyesuaian. Struktur berkasnya:

```
baris 1 : latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation
baris 2 : -6.9175,107.625,768.0,25200,Asia/Jakarta,GMT+7
baris 3 : (kosong)
baris 4 : time,temperature_2m (°C),relative_humidity_2m (%),...
baris 5+: data
```

Tiga baris pertama adalah blok metadata. Fungsi di bawah mendeteksi sendiri baris mana yang berisi header sebenarnya, membaca blok metadata sebagai informasi lokasi, lalu membersihkan satuan dari nama kolom sehingga skemanya sama persis dengan hasil penarikan via API.

Nama berkas diasumsikan mengandung nama kota, contohnya `bandung.csv` atau `open-meteo-bandung.csv`.

In [ ]:
def baca_csv_openmeteo(path: Path, nama_kota: str) -> pd.DataFrame:
    '''Baca CSV hasil unduhan manual Open-Meteo beserta blok metadatanya.'''
    baris = path.read_text(encoding="utf-8").splitlines()

    # Cari baris header data: baris pertama yang diawali "time,"
    idx_header = next(i for i, b in enumerate(baris) if b.startswith("time,"))

    # Blok metadata berada di dua baris pertama (nama kolom + nilainya)
    meta = dict(zip(baris[0].split(","), baris[1].split(",")))

    df = pd.read_csv(path, skiprows=idx_header)

    # Buang satuan dari nama kolom: "temperature_2m (°C)" -> "temperature_2m"
    df.columns = [c.split(" (")[0].strip() for c in df.columns]

    df.insert(0, "city", nama_kota)
    df["latitude"]    = float(meta["latitude"])
    df["longitude"]   = float(meta["longitude"])
    df["elevation_m"] = float(meta["elevation"])
    return df


# Contoh pemakaian — sesuaikan MANUAL_DIR dengan lokasi berkas unduhan kalian.
MANUAL_DIR = BASE_DIR / "00_unduhan_manual"

if MANUAL_DIR.exists():
    potongan_manual = []
    for path in sorted(MANUAL_DIR.glob("*.csv")):
        # Cocokkan nama berkas dengan daftar kota yang difiksasi
        kota = next((k for k in KOTA if k.lower() in path.stem.lower()), path.stem)
        d = baca_csv_openmeteo(path, kota)
        potongan_manual.append(d)
        print(f"{path.name:<35} -> {kota:<12} {d.shape[0]:>7,} baris x {d.shape[1]:>2} kolom")

    df_manual = pd.concat(potongan_manual, ignore_index=True)
    print(f"\nGabungan jalur manual: {df_manual.shape[0]:,} baris x {df_manual.shape[1]} kolom")
else:
    print(f"Folder {MANUAL_DIR} belum ada — lewati jalur manual, "
          f"gunakan hasil penarikan API pada Bagian 7.")